# 02 · Datasets, TSVs, STM, and icefall Manifests

How to point `zipa-cli` at the data formats you already have. Code cells that need
the ONNX stack are guarded; the command patterns apply to any backend/model.

In [ ]:
# Make the zipa_cli package importable when running from tutorials/ without install.
import sys, os
REPO = os.path.abspath('..')
if REPO not in sys.path: sys.path.insert(0, REPO)
from zipa_cli.cli import main as zipa  # call like zipa(['models','list'])
print('zipa-cli importable from', REPO)

In [ ]:
# Which backend deps are available in this kernel?
def have(mod):
    try:
        __import__(mod); return True
    except Exception:
        return False
ONNX_OK = all(have(m) for m in ['onnxruntime','soundfile','lhotse'])
print('ONNX inference available:', ONNX_OK)
print('(install with: pip install -e "..[onnx]")')

In [ ]:
# Build a tiny *valid* ONNX CTC model + tokens so this notebook runs end-to-end
# WITHOUT downloading the multi-hundred-MB ZIPA checkpoints. Real usage just swaps
# --model <path> for a registry tag like 'zipa-cr-s-300k'.
import numpy as np, tempfile
TMP = tempfile.mkdtemp(prefix='zipa_demo_')
def build_tiny_ctc():
    import onnx
    from onnx import helper, TensorProto
    V = 10
    W = (np.random.RandomState(0).randn(80, V)*0.1).astype(np.float32)
    g = helper.make_graph([helper.make_node('MatMul',['x','W'],['logits'])],'tiny',
        [helper.make_tensor_value_info('x',TensorProto.FLOAT,['B','T',80]),
         helper.make_tensor_value_info('x_lens',TensorProto.INT64,['B'])],
        [helper.make_tensor_value_info('logits',TensorProto.FLOAT,['B','T',V])],
        [helper.make_tensor('W',TensorProto.FLOAT,[80,V],W.flatten())])
    m = helper.make_model(g, opset_imports=[helper.make_opsetid('',13)]); m.ir_version=9
    p = os.path.join(TMP,'tiny_ctc.onnx'); onnx.save(m,p)
    tk = os.path.join(TMP,'tokens.txt')
    with open(tk,'w') as f:
        for i,t in enumerate('<blk> a b c d e f g h i'.split()): f.write(f'{t} {i}\n')
    return p, tk
MODEL = TOKENS = None
if ONNX_OK and have('onnx'):
    MODEL, TOKENS = build_tiny_ctc()
    print('demo model:', MODEL)
else:
    print('Skipping demo model (need onnxruntime+lhotse+onnx). Markdown cells still apply.')

## CommonVoice-style TSV
`--id-col/--path-col/--ref-col` accept a column name or index.

In [ ]:
import shutil
SAMPLE = os.path.join(REPO,'zipa','sample.wav')
tsv = os.path.join(TMP,'clips.tsv')
with open(tsv,'w') as f:
    f.write('client\tpath\tsentence\n')
    f.write(f'u1\t{SAMPLE}\thello world\n')
if MODEL:
    out=os.path.join(TMP,'cv.jsonl')
    zipa(['decode','--input',tsv,'--input-type','tsv','--id-col','0','--path-col','path',
          '--ref-col','sentence','--model',MODEL,'--model-type','ctc','--tokens',TOKENS,
          '-o',out,'--output-format','jsonl'])
    print(open(out).read())

## STM segments + an audio directory
`file spk chan start end [text]`. Each segment is sliced from the full audio.

In [ ]:
stm=os.path.join(TMP,'show.stm')
with open(stm,'w') as f:
    f.write('sample 1 0 0.0 2.0 first part\n')
    f.write('sample 1 0 2.0 4.0 second part\n')
adir=os.path.join(TMP,'stm_audio'); os.makedirs(adir, exist_ok=True)
shutil.copy(SAMPLE, os.path.join(adir,'sample.wav'))
if MODEL:
    out=os.path.join(TMP,'segs.tsv')
    zipa(['decode','--input',stm,'--input-type','stm','--audio-dir',adir,'--model',MODEL,
          '--model-type','ctc','--tokens',TOKENS,'-o',out])
    print(open(out).read())

## icefall / lhotse manifest
Build a CutSet from `sample.wav`, then decode it. Precomputed-feature cuts skip
fbank automatically.

In [ ]:
if ONNX_OK:
    from lhotse import Recording, RecordingSet, SupervisionSegment, SupervisionSet, CutSet
    rec = Recording.from_file(SAMPLE, recording_id='sample')
    sup = SupervisionSegment(id='sample-0', recording_id='sample', start=0.0,
                             duration=rec.duration, text='r e f', channel=0)
    cuts = CutSet.from_manifests(recordings=RecordingSet.from_recordings([rec]),
                                 supervisions=SupervisionSet.from_segments([sup]))
    cpath=os.path.join(TMP,'cuts.jsonl.gz'); cuts.to_file(cpath)
    if MODEL:
        out=os.path.join(TMP,'manifest.recogs')
        zipa(['decode','--input',cpath,'--input-type','manifest','--cuts',cpath,'--model',MODEL,
              '--model-type','ctc','--tokens',TOKENS,'-o',out,'--output-format','recogs'])
        print(open(out).read())

### HuggingFace datasets & shar
```bash
# Streaming HF dataset
zipa-cli decode --input-type hf --hf-dataset mozilla-foundation/common_voice_17_0 \
    --hf-config en --hf-split test --model zipa-cr-l-500k -o cv.jsonl --output-format jsonl

# lhotse shar shards directory
zipa-cli decode --input data-shar/ --input-type shar --model zipa-cr-l-500k -o hyp.tsv
```